# Análise Socioeconômica — PNAD COVID-19
## Como se preparar diante de uma nova pandemia?

Este estudo analisa o **perfil socioeconômico** da população entrevistada na pesquisa **PNAD COVID-19 (IBGE)** com o objetivo de identificar padrões e indicadores que auxiliem hospitais e gestores de saúde pública no **planejamento estratégico para futuros surtos pandêmicos**.

> Os dados representam um **retrato geral da população** entrevistada, sem filtro por presença ou ausência de sintomas.

As dimensões analisadas são:
- Distribuição de renda (faixas salariais)
- Categorias ocupacionais (tipo de trabalho)
- Dependência de programas de assistência social
- Acesso a itens de higiene e proteção individual

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Carregar todos os CSVs econômicos
df_salario     = pd.read_csv('csv_economicas/C10_faixa_salarial.csv')
df_trabalho    = pd.read_csv('csv_economicas/C7_tipo_trabalho.csv')
df_assistencia = pd.read_csv('csv_economicas/D1_assistencialismo.csv')
df_higiene     = pd.read_csv('csv_economicas/F2_itens_higienie.csv')

print('Dados carregados com sucesso!')

---
## 1. Distribuição de Renda da População

A faixa salarial é um dos determinantes mais relevantes na capacidade de resposta individual a uma pandemia. Populações de baixa renda enfrentam maiores dificuldades para:
- Aderir ao isolamento social (dependem de trabalho presencial)
- Adquirir insumos de proteção individual
- Acessar serviços de saúde suplementar

Compreender essa distribuição permite dimensionar a **dependência do sistema público de saúde** e a **vulnerabilidade econômica** da população.

In [ ]:
# Gráfico 1 — Distribuição por faixa salarial
fig, ax = plt.subplots(figsize=(14, 7))

cores = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(df_salario)))

bars = ax.barh(df_salario['faixa_salarial'], df_salario['total'], color=cores)

# Anotação de valores
for bar, val in zip(bars, df_salario['total']):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=10)

ax.set_title('Distribuição da População por Faixa Salarial', fontsize=16, fontweight='bold')
ax.set_xlabel('Número de Entrevistados')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2 — Concentração por grandes grupos de renda
# Agrupando em: Baixa (até 2 SM), Média (2-10 SM), Alta (acima de 10 SM)
# Salário mínimo 2020 ≈ R$ 1.045
baixa_renda  = df_salario[df_salario['codigo'].isin([1, 2, 3])]['total'].sum()
media_renda  = df_salario[df_salario['codigo'].isin([4, 5, 6, 7])]['total'].sum()
alta_renda   = df_salario[df_salario['codigo'].isin([8, 9, 10])]['total'].sum()

grupos = ['Baixa renda\n(até ~R$2.090)', 'Renda média\n(R$2.090 – R$10.451)', 'Alta renda\n(acima de R$10.451)']
totais_grupo = [baixa_renda, media_renda, alta_renda]
cores_grupo  = ['#F44336', '#FF9800', '#4CAF50']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pizza
wedges, texts, autotexts = ax1.pie(
    totais_grupo, labels=grupos, autopct='%1.1f%%',
    colors=cores_grupo, startangle=90,
    textprops={'fontsize': 11}
)
ax1.set_title('Concentração de Renda (% da população)', fontsize=13, fontweight='bold')

# Barras absolutas
bars = ax2.bar(grupos, totais_grupo, color=cores_grupo, edgecolor='white')
for bar, val in zip(bars, totais_grupo):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5000,
             f'{val:,}', ha='center', fontsize=11, fontweight='bold')
ax2.set_title('Total de Entrevistados por Grupo de Renda', fontsize=13, fontweight='bold')
ax2.set_ylabel('Número de Entrevistados')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

fig.suptitle('Estratificação de Renda — PNAD COVID-19', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

total_geral = sum(totais_grupo)
print(f'Total de entrevistados com renda informada: {total_geral:,}')
print(f'Baixa renda: {baixa_renda/total_geral*100:.1f}%')
print(f'Renda média: {media_renda/total_geral*100:.1f}%')
print(f'Alta renda:  {alta_renda/total_geral*100:.1f}%')

### Insight — Renda

A maior concentração de entrevistados se encontra nas faixas de **baixa e média-baixa renda**, com destaque para a faixa de R$ 801 a R$ 1.600. Isso evidencia uma população altamente dependente do sistema público de saúde e com limitada capacidade de isolamento voluntário.

**Recomendação para preparação:**
- Políticas de **renda emergencial** são essenciais para viabilizar o isolamento social
- O sistema público de saúde deve ser dimensionado para absorver a **grande maioria** da demanda
- Estratégias de comunicação devem considerar populações de baixa escolaridade e renda

---
## 2. Perfil Ocupacional — Tipos de Trabalho

A categoria profissional determina diretamente o **nível de exposição ao vírus** e a **capacidade de adoção de trabalho remoto**. Trabalhadores essenciais e de serviços não podem aderir ao isolamento, elevando o risco de transmissão comunitária.

In [ ]:
# Gráfico 3 — Top 15 categorias ocupacionais (excluindo 'Não especificado')
df_trab_filtrado = df_trabalho[df_trabalho['descricao_trabalho'] != 'Não especificado'].copy()
df_trab_top15 = df_trab_filtrado.nlargest(15, 'total')

fig, ax = plt.subplots(figsize=(14, 8))
cores_trab = plt.cm.Blues(np.linspace(0.4, 0.9, len(df_trab_top15)))

bars = ax.barh(df_trab_top15['descricao_trabalho'], df_trab_top15['total'], color=cores_trab)

for bar, val in zip(bars, df_trab_top15['total']):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=9)

ax.set_title('Top 15 Categorias Ocupacionais — Entrevistados PNAD COVID-19', fontsize=15, fontweight='bold')
ax.set_xlabel('Número de Entrevistados')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 4 — Classificação por nível de exposição ao risco
# Categorias de alto risco: saúde, segurança, transporte, serviços essenciais
alto_risco_keywords = [
    'saúde', 'médic', 'enfermeir', 'farmac', 'dentist',
    'policiais', 'bombeiro', 'seguranç',
    'doméstic', 'limpeza', 'faxineiro',
    'motorist', 'entregador', 'caminhoneir',
    'comerci', 'vendedor', 'caixa', 'atendente',
    'agricultor', 'lavoura', 'pecuária'
]

remoto_keywords = [
    'professor', 'docente', 'administrat', 'gerente', 'diretor',
    'engenheiro', 'analista', 'programador', 'contador', 'advogado',
    'economista', 'arquiteto', 'jornalista', 'pesquisador'
]

def classificar(descricao):
    desc_lower = descricao.lower()
    for kw in alto_risco_keywords:
        if kw in desc_lower:
            return 'Alto risco / presencial obrigatório'
    for kw in remoto_keywords:
        if kw in desc_lower:
            return 'Possível trabalho remoto'
    return 'Outros'

df_trabalho['classificacao'] = df_trabalho['descricao_trabalho'].apply(classificar)
resumo_class = df_trabalho.groupby('classificacao')['total'].sum().reset_index()

cores_class = {'Alto risco / presencial obrigatório': '#F44336',
               'Possível trabalho remoto': '#4CAF50',
               'Outros': '#9E9E9E'}

fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    resumo_class['total'],
    labels=resumo_class['classificacao'],
    autopct='%1.1f%%',
    colors=[cores_class[c] for c in resumo_class['classificacao']],
    startangle=90,
    textprops={'fontsize': 11}
)
ax.set_title('Perfil Ocupacional por Nível de Exposição ao Risco', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

for _, row in resumo_class.iterrows():
    print(f"{row['classificacao']}: {row['total']:,}")

### Insight — Ocupação

Uma parcela significativa da população exerce atividades **presenciais obrigatórias**, com destaque para trabalhadores de serviços domésticos, comércio, saúde e segurança. Essas categorias não podem aderir ao isolamento social, tornando-se vetores involuntários de transmissão.

**Recomendação para preparação:**
- Criar **protocolos específicos por categoria profissional** para minimizar exposição
- Priorizar **trabalhadores essenciais** em campanhas de vacinação e testagem
- Investir em **EPIs gratuitos** para trabalhadores de baixa renda em atividades presenciais
- Mapear os setores de alto risco como **pontos de vigilância epidemiológica ativa**

---
## 3. Dependência de Assistência Social

A proporção de entrevistados que recebe algum tipo de benefício social revela o **nível de vulnerabilidade econômica** da população. Em contextos de pandemia, essa população é a mais impactada pela perda de emprego e renda, aumentando a demanda por serviços públicos de saúde.

In [ ]:
# Gráfico 5 — Número de beneficiários por tipo de assistência
beneficios = {
    'Aposentadoria / Pensão':  df_assistencia['aposentadoria_pensao'].values[0],
    'Pensão Alimentícia':      df_assistencia['pensao_alimenticia'].values[0],
    'Bolsa Família':           df_assistencia['bolsa_familia'].values[0],
    'Benefício Assistencial\n(BPC/LOAS)': df_assistencia['beneficio_assistencial'].values[0],
    'Auxílio Emergencial\n(COVID-19)': df_assistencia['auxilio_emergencial'].values[0],
    'Seguro Desemprego':       df_assistencia['desemprego_seguro'].values[0],
}

labels_b = list(beneficios.keys())
valores_b = list(beneficios.values())
cores_b = ['#5C6BC0', '#AB47BC', '#26A69A', '#EF5350', '#FFA726', '#66BB6A']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(labels_b, valores_b, color=cores_b, edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, valores_b):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8000,
            f'{val:,}', ha='center', fontsize=11, fontweight='bold')

ax.set_title('Beneficiários de Programas de Assistência Social', fontsize=16, fontweight='bold')
ax.set_ylabel('Número de Entrevistados')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Comparativo: auxílio emergencial vs demais benefícios permanentes
total_permanentes = (beneficios['Aposentadoria / Pensão'] +
                     beneficios['Pensão Alimentícia'] +
                     beneficios['Bolsa Família'] +
                     beneficios['Benefício Assistencial\n(BPC/LOAS)'] +
                     beneficios['Seguro Desemprego'])
auxilio_emerg = beneficios['Auxílio Emergencial\n(COVID-19)']

fig, ax = plt.subplots(figsize=(8, 6))
categorias = ['Benefícios\nPermanentes', 'Auxílio\nEmergencial (COVID)']
totais_comp = [total_permanentes, auxilio_emerg]
cores_comp  = ['#5C6BC0', '#FFA726']

bars = ax.bar(categorias, totais_comp, color=cores_comp, width=0.5, edgecolor='white')
for bar, val in zip(bars, totais_comp):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10000,
            f'{val:,}', ha='center', fontsize=13, fontweight='bold')

ax.set_title('Benefícios Permanentes vs Auxílio Emergencial', fontsize=14, fontweight='bold')
ax.set_ylabel('Número de Beneficiários')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

print(f'Total em benefícios permanentes: {total_permanentes:,}')
print(f'Auxílio emergencial COVID-19:     {auxilio_emerg:,}')
print(f'Razão (emergencial / permanentes): {auxilio_emerg/total_permanentes:.2f}x')

### Insight — Assistência Social

O **Auxílio Emergencial COVID-19** foi o benefício mais recebido, superando a soma dos benefícios permanentes. Isso evidencia que a pandemia gerou uma nova camada de vulnerabilidade econômica que não existia antes do surto.

**Recomendação para preparação:**
- Estruturar **mecanismos ágeis de transferência de renda emergencial** previamente ao surto, para ativação rápida
- Integrar os cadastros de assistência social com os sistemas de saúde para **identificar populações vulneráveis** em tempo real
- Reforçar serviços de saúde em regiões com alta concentração de beneficiários do Bolsa Família e BPC

---
## 4. Acesso a Itens de Higiene e Proteção Individual

O acesso a insumos básicos de proteção — sabão, álcool 70%, máscaras, luvas e água sanitária — é determinante para a capacidade de prevenção individual. A ausência desses itens representa uma **falha crítica de infraestrutura sanitária** que amplifica a transmissão comunitária.

In [ ]:
# Gráfico 7 — Acesso a itens de higiene: sim / não / não sabe
itens = {
    'Sabão /\nDetergente': {
        'Sim': df_higiene['sabao_detergente_sim'].values[0],
        'Não': df_higiene['sabao_detergente_nao'].values[0],
        'Não sabe': df_higiene['sabao_detergente_nao_sabe'].values[0]
    },
    'Álcool 70%': {
        'Sim': df_higiene['alcool_setenta_sim'].values[0],
        'Não': df_higiene['alcool_setenta_nao'].values[0],
        'Não sabe': df_higiene['alcool_setenta_nao_sabe'].values[0]
    },
    'Máscaras': {
        'Sim': df_higiene['mascaras_sim'].values[0],
        'Não': df_higiene['mascaras_nao'].values[0],
        'Não sabe': df_higiene['mascaras_nao_sabe'].values[0]
    },
    'Luvas': {
        'Sim': df_higiene['luvas_sim'].values[0],
        'Não': df_higiene['luvas_nao'].values[0],
        'Não sabe': df_higiene['luvas_nao_sabe'].values[0]
    },
    'Água Sanitária': {
        'Sim': df_higiene['agua_sanitaria_sim'].values[0],
        'Não': df_higiene['agua_sanitaria_nao'].values[0],
        'Não sabe': df_higiene['agua_sanitaria_nao_sabe'].values[0]
    },
}

labels_itens = list(itens.keys())
sim_vals     = [itens[i]['Sim'] for i in labels_itens]
nao_vals     = [itens[i]['Não'] for i in labels_itens]
ns_vals      = [itens[i]['Não sabe'] for i in labels_itens]

x = np.arange(len(labels_itens))
w = 0.28

fig, ax = plt.subplots(figsize=(13, 7))
ax.bar(x - w, sim_vals, w, label='Possui', color='#4CAF50')
ax.bar(x,     nao_vals, w, label='Não possui', color='#F44336')
ax.bar(x + w, ns_vals,  w, label='Não sabe', color='#BDBDBD')

ax.set_xticks(x)
ax.set_xticklabels(labels_itens, fontsize=12)
ax.set_title('Acesso a Itens de Higiene e Proteção Individual', fontsize=16, fontweight='bold')
ax.set_ylabel('Número de Entrevistados')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 8 — Percentual de acesso (Sim) para cada item
totais_por_item = {
    label: itens[label]['Sim'] + itens[label]['Não'] + itens[label]['Não sabe']
    for label in labels_itens
}
pct_acesso = [itens[label]['Sim'] / totais_por_item[label] * 100 for label in labels_itens]
pct_nao    = [itens[label]['Não'] / totais_por_item[label] * 100 for label in labels_itens]

cores_itens = ['#4CAF50' if p >= 90 else '#FF9800' if p >= 60 else '#F44336' for p in pct_acesso]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(labels_itens, pct_acesso, color=cores_itens, edgecolor='white', linewidth=1.5)

# Linha de referência 90%
ax.axhline(90, color='gray', linestyle='--', linewidth=1.2, label='90% de referência')

for bar, pct, np_val in zip(bars, pct_acesso, pct_nao):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', fontsize=12, fontweight='bold')
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
            f'Sem: {np_val:.1f}%', ha='center', fontsize=9, color='white', fontweight='bold')

ax.set_title('Percentual de Acesso a Itens de Proteção', fontsize=16, fontweight='bold')
ax.set_ylabel('% com acesso')
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.show()

print('Itens com maior lacuna de acesso:')
for label, pct in sorted(zip(labels_itens, pct_nao), key=lambda x: -x[1]):
    print(f'  {label.replace(chr(10), " ")}: {pct:.1f}% sem acesso')

### Insight — Higiene e Proteção

Enquanto **sabão, álcool 70%, máscaras e água sanitária** apresentam altíssimos índices de acesso (>95%), as **luvas** são o item com maior déficit, acessíveis a menos de 40% da população. Isso indica que a proteção individual foi relativamente bem disseminada para EPIs de uso geral, mas itens de proteção mais específicos permanecem restritos.

**Recomendação para preparação:**
- Manter **reserva estratégica de insumos de higiene** (especialmente álcool 70% e máscaras) para distribuição em surtos
- O alto índice de acesso a itens básicos sugere que **campanhas de distribuição de máscaras e álcool** foram eficazes — replicar esse modelo
- Luvas devem ser priorizadas para **trabalhadores presenciais de alto risco**, não necessariamente para toda a população

---
## 5. Dashboard Consolidado — Vulnerabilidade Socioeconômica

In [ ]:
# Gráfico 9 — Dashboard consolidado com os 4 indicadores-chave
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top-left: Top 8 faixas salariais
ax = axes[0, 0]
df_sal_top8 = df_salario.nlargest(8, 'total')
cores_sal = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(df_sal_top8)))
ax.barh(df_sal_top8['faixa_salarial'], df_sal_top8['total'], color=cores_sal)
ax.set_title('Distribuição Salarial (top 8)', fontweight='bold', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.invert_yaxis()

# Top-right: Assistência social
ax = axes[0, 1]
labels_assist = ['Aposent./\nPensão', 'Bolsa\nFamília', 'BPC/\nLOAS', 'Auxílio\nEmerg.', 'Seg.\nDesemprego']
vals_assist   = [
    df_assistencia['aposentadoria_pensao'].values[0],
    df_assistencia['bolsa_familia'].values[0],
    df_assistencia['beneficio_assistencial'].values[0],
    df_assistencia['auxilio_emergencial'].values[0],
    df_assistencia['desemprego_seguro'].values[0],
]
cores_assist = ['#5C6BC0', '#26A69A', '#AB47BC', '#FFA726', '#66BB6A']
ax.bar(labels_assist, vals_assist, color=cores_assist)
ax.set_title('Beneficiários por Programa Social', fontweight='bold', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Bottom-left: Acesso a EPIs (% com acesso)
ax = axes[1, 0]
cores_epi = ['#4CAF50' if p >= 90 else '#FF9800' if p >= 60 else '#F44336' for p in pct_acesso]
labels_itens_curto = ['Sabão', 'Álcool 70%', 'Máscaras', 'Luvas', 'Água\nSanitária']
ax.bar(labels_itens_curto, pct_acesso, color=cores_epi)
ax.axhline(90, color='gray', linestyle='--', linewidth=1)
ax.set_title('Acesso a EPIs (%)', fontweight='bold', fontsize=12)
ax.set_ylabel('%')
ax.set_ylim(0, 110)
for i, p in enumerate(pct_acesso):
    ax.text(i, p + 1, f'{p:.0f}%', ha='center', fontsize=9)

# Bottom-right: Perfil de exposição ocupacional
ax = axes[1, 1]
cores_ocup = [cores_class[c] for c in resumo_class['classificacao']]
wedges, texts, autotexts = ax.pie(
    resumo_class['total'],
    labels=resumo_class['classificacao'],
    autopct='%1.0f%%',
    colors=cores_ocup,
    startangle=90,
    textprops={'fontsize': 10}
)
ax.set_title('Exposição Ocupacional ao Risco', fontweight='bold', fontsize=12)

fig.suptitle('Dashboard — Vulnerabilidade Socioeconômica (PNAD COVID-19)', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Conclusão — Plano Socioeconômico para Preparação Pandêmica

A análise do perfil socioeconômico da população entrevistada pela PNAD COVID-19 revela **quatro vetores críticos de vulnerabilidade** que devem orientar o planejamento estratégico de hospitais e gestores de saúde pública:

### 1. Renda e Capacidade de Isolamento
- A maioria da população está concentrada em **faixas de baixa renda**, incapaz de arcar com serviços privados de saúde
- A capacidade de isolamento social depende de **suporte financeiro emergencial** — sem renda substituta, o isolamento é inviável
- **Ação:** Pré-estruturar mecanismos de pagamento de auxílio emergencial para ativação em até 30 dias do início de um surto

### 2. Perfil Ocupacional e Exposição
- Trabalhadores de serviços, comércio, saúde e segurança pública **não podem aderir ao isolamento**
- Esses grupos devem ser tratados como **populações prioritárias** em vacinação, testagem e fornecimento de EPIs
- **Ação:** Mapear setores essenciais por município e criar protocolos específicos de proteção por categoria profissional

### 3. Assistência Social como Termômetro de Crise
- O volume de beneficiários do Auxílio Emergencial supera o de todos os programas permanentes combinados
- Isso sinaliza que pandemias **criam novas camadas de pobreza** além das já existentes
- **Ação:** Integrar dados de assistência social com vigilância epidemiológica para identificar regiões de maior vulnerabilidade

### 4. Insumos de Proteção como Infraestrutura de Saúde Pública
- O alto acesso a máscaras e álcool 70% mostra que **distribuição emergencial é viável e eficaz**
- Luvas permanecem subacessíveis — devem ser direcionadas a trabalhadores de risco
- **Ação:** Manter reserva estratégica nacional de insumos básicos de proteção individual para distribuição imediata no início de surtos

---
*Estudo desenvolvido com dados da PNAD COVID-19 (IBGE) — Projeto FIAP*